# **06. Đánh giá Mô hình & Phân tích Chiều sâu Kinh doanh (Business Intelligence)**

## **Mục tiêu:**
- Rà soát chất lượng dự báo của mô hình trên tập dữ liệu.
- Trực quan hóa phân bổ các khía cạnh và cảm xúc.
- Tạo ma trận chéo biểu diễn mối quan hệ **Khía cạnh x Cảm xúc** (Heatmap).
- Định vị các điểm nghẽn vận hành (Vận chuyển, Dịch vụ) của nhãn hàng Vinamilk.
- Vẽ biểu đồ đường thể hiện xu hướng tương tác và cảm xúc khách hàng theo dòng thời gian.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sys.path.append(os.path.abspath('../src'))
import utils
from labeling import correct_aspect_typos

### **1. Nạp dữ liệu huấn luyện để trực quan hóa**
Chúng ta tải tập dữ liệu huấn luyện đã xử lý xong để thực hiện thống kê.

In [ ]:
train_dataset_path = "../data/processed/train_dataset.csv"
df = pd.read_csv(train_dataset_path, encoding="utf-8-sig")
df = correct_aspect_typos(df)
print(f"Loaded {len(df):,} reviews for visualization.")

### **2. Trực quan hóa Phân bổ Cảm xúc (Pie & Bar Chart)**
Biểu đồ thể hiện tỷ lệ của 3 lớp cảm xúc.

In [ ]:
sentiment_map = {0: "Tiêu cực (0)", 1: "Trung lập (1)", 2: "Tích cực (2)"}
labeled_sentiment = df['LLM_Sentiment'].map(sentiment_map)
sentiment_counts = labeled_sentiment.value_counts()

plt.figure(figsize=(14, 5))

# Bar Chart
plt.subplot(1, 2, 1)
order_list = ["Tiêu cực (0)", "Trung lập (1)", "Tích cực (2)"]
sns.barplot(
    x=sentiment_counts.index,
    y=sentiment_counts.values,
    order=order_list,
    palette=['#e74c3c', '#f1c40f', '#2ecc71'],
    hue=sentiment_counts.index,
    legend=False
)
plt.title('Số lượng bình luận theo nhóm cảm xúc (Tập cân bằng)', fontsize=12, fontweight='bold', pad=15)
plt.ylabel('Số lượng (Reviews)', fontsize=10)
plt.xlabel('Cảm xúc (Sentiment)', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Pie Chart
plt.subplot(1, 2, 2)
colors_pie = ['#2ecc71', '#f1c40f', '#e74c3c'] if sentiment_counts.index[0] == "Tích cực (2)" else ['#e74c3c', '#f1c40f', '#2ecc71']
plt.pie(
    sentiment_counts.values,
    labels=sentiment_counts.index,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors_pie,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
plt.title('Tỷ lệ phân bổ cảm xúc trong tập huấn luyện', fontsize=12, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig("../outputs/figures/sentiment_distribution.png", dpi=300)
plt.show()

### **3. Trực quan hóa Phân bổ các Khía cạnh (Aspects Distribution)**

In [ ]:
all_aspects = []
for aspect_cell in df["LLM_Aspect"].dropna():
    parts = [a.strip() for a in str(aspect_cell).split(",") if a.strip()]
    all_aspects.extend(parts)

aspect_counts = Counter(all_aspects)
aspect_df = pd.DataFrame(aspect_counts.most_common(), columns=["Khía cạnh", "Số lượt xuất hiện"])
aspect_df["Tỷ lệ (%)"] = (aspect_df["Số lượt xuất hiện"] / len(df)) * 100

plt.figure(figsize=(10, 6))
ax = sns.barplot(
    x="Số lượt xuất hiện",
    y="Khía cạnh",
    data=aspect_df,
    palette="Blues_r",
    hue="Khía cạnh",
    legend=False
)

for i, patch in enumerate(ax.patches):
    current_width = patch.get_width()
    current_y = patch.get_y()
    current_height = patch.get_height()
    count = aspect_df.iloc[i]["Số lượt xuất hiện"]
    pct = aspect_df.iloc[i]["Tỷ lệ (%)"]
    ax.text(
        current_width + (max(aspect_df["Số lượt xuất hiện"]) * 0.01),
        current_y + current_height/2,
        f"{count:,} ({pct:.1f}%)",
        va='center', ha='left', fontsize=10, fontweight='bold', color='#2c3e50'
    )

plt.title('Phân bổ các khía cạnh (Aspect) trong tập dữ liệu huấn luyện', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Số lượt xuất hiện (Lần)', fontsize=11, labelpad=10)
plt.ylabel('Danh mục khía cạnh', fontsize=11, labelpad=10)
plt.xlim(0, max(aspect_df["Số lượt xuất hiện"]) * 1.15)
plt.grid(axis='x', linestyle='--', alpha=0.5)
sns.despine(left=True, bottom=True)

plt.tight_layout()
plt.savefig("../outputs/figures/aspect_distribution.png", dpi=300)
plt.show()

### **4. Ma trận phân bổ Cảm xúc × Khía cạnh (Heatmap)**
Tính toán chéo tỷ lệ cảm xúc tương quan với từng khía cạnh sản phẩm chính.

In [ ]:
df_exploded = df.copy()
df_exploded['LLM_Aspect'] = df_exploded['LLM_Aspect'].astype(str).str.split(',')
df_exploded = df_exploded.explode('LLM_Aspect')
df_exploded['LLM_Aspect'] = df_exploded['LLM_Aspect'].str.strip()

# Chỉ lấy các khía cạnh chính
main_aspects = ['Chất lượng', 'Giao hàng', 'Dịch vụ', 'Giá cả', 'Đúng mô tả']
df_exploded = df_exploded[df_exploded['LLM_Aspect'].isin(main_aspects)]

aspect_sentiment_matrix = pd.crosstab(
    df_exploded['LLM_Aspect'],
    df_exploded['LLM_Sentiment'],
    normalize='index'
) * 100

aspect_sentiment_matrix.columns = ['Tiêu cực (0)', 'Trung lập (1)', 'Tích cực (2)']

plt.figure(figsize=(10, 6))
sns.heatmap(
    aspect_sentiment_matrix,
    annot=True,
    fmt=".1f",
    cmap="YlGnBu",
    linewidths=0.5,
    annot_kws={"size": 11, "weight": "bold"},
    cbar_kws={'label': 'Tỷ lệ phần trăm (%)'}
)

plt.title('Ma trận phân bổ Cảm xúc theo từng Khía cạnh sản phẩm', fontsize=13, fontweight='bold', pad=20)
plt.ylabel('Các khía cạnh phân tích (Aspect)', fontsize=11, labelpad=10)
plt.xlabel('Trạng thái cảm xúc (Sentiment)', fontsize=11, labelpad=10)

plt.tight_layout()
plt.savefig("../outputs/figures/aspect_sentiment_heatmap.png", dpi=300)
plt.show()

### **5. Phân tích Cảm xúc theo mã Sản phẩm (Product Level Analysis)**
Tìm ra Top 5 sản phẩm được yêu thích và Top 5 sản phẩm nhận nhiều phản hồi tiêu cực nhất để doanh nghiệp cải thiện.

In [ ]:
product_sentiment = pd.crosstab(
    df['product_title'],
    df['LLM_Sentiment'],
    normalize='index'
) * 100

product_counts = df['product_title'].value_counts()
# Lọc các sản phẩm có từ 1 đánh giá trở lên
popular_products = product_counts[product_counts >= 1].index
filtered_product_sentiment = product_sentiment.loc[product_sentiment.index.isin(popular_products)]

# Top 5 được yêu thích (Lớp 2: Tích cực cao nhất)
top_loved = filtered_product_sentiment.sort_values(by=2, ascending=False).head(5)
# Top 5 tiêu cực (Lớp 0: Tiêu cực cao nhất)
top_complaints = filtered_product_sentiment.sort_values(by=0, ascending=False).head(5)

plt.figure(figsize=(14, 10))

# Top Loved Bar Chart
plt.subplot(2, 1, 1)
short_loved_labels = [f"{prod[:50]}..." for prod in top_loved.index]
ax1 = sns.barplot(
    x=top_loved[2],
    y=short_loved_labels,
    palette='crest',
    hue=short_loved_labels,
    legend=False
)
plt.title('Top 5 sản phẩm có tỷ lệ phản hồi Tích cực (%) cao nhất', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Tỷ lệ Tích cực (%)', fontsize=10)
plt.xlim(0, 115)

for patch in ax1.patches:
    width = patch.get_width()
    if width > 0:
        ax1.text(
            width + 1,
            patch.get_y() + patch.get_height()/2,
            f"{width:.1f}%",
            va='center', ha='left', fontsize=10, fontweight='bold', color='#2c3e50'
        )

# Top Complaints Bar Chart
plt.subplot(2, 1, 2)
short_complaints_labels = [f"{prod[:50]}..." for prod in top_complaints.index]
ax2 = sns.barplot(
    x=top_complaints[0],
    y=short_complaints_labels,
    palette='flare',
    hue=short_complaints_labels,
    legend=False
)
plt.title('Top 5 sản phẩm có tỷ lệ phản hồi Tiêu cực (%) cao nhất', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Tỷ lệ Tiêu cực (%)', fontsize=10)

max_neg = top_complaints[0].max()
plt.xlim(0, max_neg * 1.15 if max_neg > 0 else 10)

for patch in ax2.patches:
    width = patch.get_width()
    if width > 0:
        ax2.text(
            width + (max_neg * 0.01 if max_neg > 0 else 0.1),
            patch.get_y() + patch.get_height()/2,
            f"{width:.1f}%",
            va='center', ha='left', fontsize=10, fontweight='bold', color='#2c3e50'
        )

plt.tight_layout()
plt.savefig("../outputs/figures/product_level_analysis.png", dpi=300)
plt.show()

### **6. Biến động cảm xúc theo Thời gian (Timeline Trend)**
Theo dõi xu hướng các lớp cảm xúc theo từng tháng từ dữ liệu lịch sử.

In [ ]:
df_time = df.copy()
df_time['time'] = pd.to_datetime(df_time['time'], errors='coerce')
df_time = df_time.dropna(subset=['time'])
df_time['Year_Month'] = df_time['time'].dt.to_period('M')

monthly_sentiment = pd.crosstab(
    df_time['Year_Month'],
    df_time['LLM_Sentiment']
)

monthly_sentiment.columns = ['Tiêu cực', 'Trung lập', 'Tích cực']
monthly_sentiment.index = monthly_sentiment.index.astype(str)
monthly_sentiment = monthly_sentiment.sort_index()

plt.figure(figsize=(14, 6))
plt.plot(monthly_sentiment.index, monthly_sentiment['Tích cực'], marker='o', color='#2ecc71', linewidth=2.5, label='Tích cực (2)')
plt.plot(monthly_sentiment.index, monthly_sentiment['Trung lập'], marker='s', color='#f1c40f', linewidth=2.0, linestyle='--', label='Trung lập (1)')
plt.plot(monthly_sentiment.index, monthly_sentiment['Tiêu cực'], marker='x', color='#e74c3c', linewidth=2.5, label='Tiêu cực (0)')

plt.title('Xu hướng biến động lượng phản hồi của khách hàng theo thời gian', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Giai đoạn (Tháng/Năm)', fontsize=11, labelpad=10)
plt.ylabel('Số lượng bình luận (Reviews)', fontsize=11, labelpad=10)
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(fontsize=11, loc='upper left')

plt.tight_layout()
plt.savefig("../outputs/figures/sentiment_timeline_trend.png", dpi=300)
plt.show()